# Insert Data into a Dataverse Table

A simple notebook that inserts a small amount of data into a Dataverse table using the **Web API (REST)**.

**Why REST?** For small volumes of data, the [Dataverse Web API](https://learn.microsoft.com/power-apps/developer/data-platform/webapi/create-entity-web-api) is the recommended approach: send a `POST` request to the entity set resource (e.g. `/api/data/v9.2/accounts`). Each successful create returns `HTTP 204 No Content` with the new record URI in the `OData-EntityId` header.

> For larger, atomic batches you would use a `$batch` request, but plain POSTs are the simplest option for a handful of records.

In [ ]:
import requests
import notebookutils

In [ ]:
# --- Configuration ---
dataverse_env_url = "https://org0f651234be2.crm4.dynamics.com"

tenant_id  = "9e929790-272d-4977-a2ab-301443c11ece"
client_id  = "b5c04c9c-0588-418f-8f60-2d83d38cb635"

azure_key_vault_name = "ab-fabric-automation-akv"
secret_name          = "fabric-monit-secret"

# Entity set name of the target Dataverse table (plural). Change to your table.
entity_set_name = "accounts"
api_version     = "v9.2"

In [ ]:
# --- Get the service principal secret from Key Vault ---
azure_key_vault_url = f"https://{azure_key_vault_name}.vault.azure.net"
client_secret = notebookutils.credentials.getSecret(azure_key_vault_url, secret_name)

In [ ]:
# --- Acquire a Dataverse access token (client credentials / SPN) ---
def get_dataverse_token():
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "scope":         dataverse_env_url.rstrip("/") + "/.default",
    }
    response = requests.post(url, data=payload)
    response.raise_for_status()
    return response.json()["access_token"]

dataverse_token = get_dataverse_token()
print("Token acquired.")

In [ ]:
# --- Insert a single record via POST ---
def insert_record(record):
    url = f"{dataverse_env_url.rstrip('/')}/api/data/{api_version}/{entity_set_name}"
    headers = {
        "Authorization":    f"Bearer {dataverse_token}",
        "OData-MaxVersion": "4.0",
        "OData-Version":    "4.0",
        "Accept":           "application/json",
        "Content-Type":     "application/json",
    }
    response = requests.post(url, headers=headers, json=record)
    response.raise_for_status()
    # On success Dataverse returns 204 with the new record URI in this header
    return response.headers.get("OData-EntityId")

In [ ]:
# --- Sample data to insert (change to match your table's columns) ---
records = [
    {"name": "Sample Account 1", "address1_city": "Redmond"},
    {"name": "Sample Account 2", "address1_city": "Seattle"},
]

for record in records:
    entity_id = insert_record(record)
    print(f"Created: {entity_id}")